# Random Forest Project

For this project we will be exploring publicly available data from [LendingClub.com](https://www.lendingclub.com/). Lending Club connects people who need money (borrowers) with people who have money (investors). Hopefully, as an investor you would want to invest in people who showed a profile of having a high probability of paying you back. We will try to create a model that will help predict this.

We will use lending data from 2007-2010 and be trying to classify and predict whether or not the borrower paid back their loan in full.

Here are what the columns represent:
* **credit.policy**: 1 if the customer meets the credit underwriting criteria of LendingClub.com, 0 otherwise.
* **purpose**: The purpose of the loan (credit_card, debt_consolidation, educational, major_purchase, small_business, all_other).
* **int.rate**: The interest rate of the loan as a proportion.
* **installment**: Monthly installments owed by the borrower.
* **log.annual.inc**: Natural log of the self-reported annual income of the borrower.
* **dti**: Debt-to-income ratio.
* **fico**: FICO credit score of the borrower.
* **days.with.cr.line**: Number of days the borrower has had a credit line.
* **revol.bal**: Borrower's revolving balance.
* **revol.util**: Revolving line utilization rate.
* **inq.last.6mths**: Number of creditor inquiries in the last 6 months.
* **delinq.2yrs**: Number of times 30+ days past due in the past 2 years.
* **pub.rec**: Number of derogatory public records.
* **not.fully.paid**: **Target** — 1 if loan was not paid back in full, 0 otherwise.

# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## Get the Data

** Use pandas to read loan_data.csv as a dataframe called loans.**

In [ ]:
loans = pd.read_csv('loan_data.csv')

** Check out the info(), head(), and describe() methods on loans.**

In [ ]:
loans.info()

In [ ]:
loans.describe().round(4)

In [ ]:
loans.head()

# Exploratory Data Analysis

** Create a histogram of two FICO distributions on top of each other, one for each credit.policy outcome.**

In [ ]:
plt.figure(figsize=(10, 5))
loans[loans['credit.policy'] == 1]['fico'].hist(
    bins=35, color='blue', label='Credit Policy = 1', alpha=0.6)
loans[loans['credit.policy'] == 0]['fico'].hist(
    bins=35, color='red',  label='Credit Policy = 0', alpha=0.6)
plt.legend()
plt.xlabel('FICO Score')
plt.ylabel('Count')
plt.title('FICO Distribution by Credit Policy')
plt.tight_layout()
plt.show()

** Create a similar figure, except this time select by the not.fully.paid column.**

In [ ]:
plt.figure(figsize=(10, 5))
loans[loans['not.fully.paid'] == 1]['fico'].hist(
    bins=35, color='blue', label='Not Fully Paid = 1', alpha=0.6)
loans[loans['not.fully.paid'] == 0]['fico'].hist(
    bins=35, color='red',  label='Not Fully Paid = 0', alpha=0.6)
plt.legend()
plt.xlabel('FICO Score')
plt.ylabel('Count')
plt.title('FICO Distribution by Not Fully Paid')
plt.tight_layout()
plt.show()

** Create a countplot using seaborn showing the counts of loans by purpose, with the color hue defined by not.fully.paid. **

In [ ]:
plt.figure(figsize=(11, 7))
sns.countplot(x='purpose', data=loans, hue='not.fully.paid', palette='Set1')
plt.xticks(rotation=30, ha='right')
plt.title('Loan Count by Purpose (coloured by Not Fully Paid)')
plt.tight_layout()
plt.show()

** Let's see the trend between FICO score and interest rate. Recreate the following jointplot.**

In [ ]:
sns.jointplot(x='fico', y='int.rate', data=loans, color='purple')
plt.show()

** Create the following lmplots to see if the trend differed between not.fully.paid and credit.policy. **

In [ ]:
sns.lmplot(y='int.rate', x='fico', data=loans,
           hue='credit.policy', col='not.fully.paid',
           palette='Set1', height=5)
plt.show()

# Setting up the Data

**Check loans.info() again.**

In [ ]:
loans.info()

## Categorical Features

Notice that the **purpose** column is categorical. We need to transform it using dummy variables so sklearn will be able to understand them.

**Create a list of 1 element containing the string 'purpose'. Call this list cat_feats.**

In [ ]:
cat_feats = ['purpose']

**Now use pd.get_dummies(loans, columns=cat_feats, drop_first=True) to create a fixed larger dataframe. Set this dataframe as final_data.**

In [ ]:
final_data = pd.get_dummies(loans, columns=cat_feats, drop_first=True)

In [ ]:
final_data.info()

## Train Test Split

** Use sklearn to split your data into a training set and a testing set.**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = final_data.drop('not.fully.paid', axis=1)
y = final_data['not.fully.paid']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=101)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')

## Training a Decision Tree Model

** Import DecisionTreeClassifier**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

**Create an instance of DecisionTreeClassifier() called dtree and fit it to the training data.**

In [ ]:
dtree = DecisionTreeClassifier()

In [ ]:
dtree.fit(X_train, y_train)

## Predictions and Evaluation of Decision Tree
**Create predictions from the test set and create a classification report and a confusion matrix.**

In [ ]:
predictions = dtree.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
print(classification_report(y_test, predictions))

In [ ]:
print(confusion_matrix(y_test, predictions))
print()
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, predictions),
    display_labels=['Fully Paid', 'Not Fully Paid'])
disp.plot(cmap='Blues')
plt.title('Decision Tree — Confusion Matrix')
plt.tight_layout()
plt.show()

## Training the Random Forest Model

**Create an instance of RandomForestClassifier and fit it to the training data.**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rfc = RandomForestClassifier(n_estimators=600, random_state=101)

In [ ]:
rfc.fit(X_train, y_train)

## Predictions and Evaluation

** Predict the class of not.fully.paid for the X_test data.**

In [ ]:
rfc_pred = rfc.predict(X_test)

**Now create a classification report from the results. Do you get anything strange or some sort of warning?**

In [ ]:
print(classification_report(y_test, rfc_pred))

**Show the Confusion Matrix for the predictions.**

In [ ]:
print(confusion_matrix(y_test, rfc_pred))
print()
disp2 = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, rfc_pred),
    display_labels=['Fully Paid', 'Not Fully Paid'])
disp2.plot(cmap='Greens')
plt.title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.show()

## Feature Importances

In [ ]:
feat_imp = pd.Series(rfc.feature_importances_,
                     index=X.columns).sort_values(ascending=False)

print('Top 10 Feature Importances:')
print(feat_imp.head(10).round(4))

feat_imp.head(10).sort_values().plot(
    kind='barh', figsize=(9, 5),
    color='steelblue', edgecolor='black')
plt.title('Random Forest — Top 10 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

**What performed better — the Random Forest or the Decision Tree?**

## Answer

The results show an interesting trade-off:

| Model | Accuracy | Class-1 Recall | Class-1 Precision |
|---|---|---|---|
| **Decision Tree** | ~72% | ~20% | ~16% |
| **Random Forest** | ~85% | ~1% | ~56% |

**Overall accuracy:** Random Forest wins (~85% vs ~72%).

**However**, the Random Forest almost entirely fails to recall class 1 (`not.fully.paid = 1`) — it predicts nearly everyone as class 0. This is the "strange warning" — the model is biased toward the majority class. The class imbalance (~16% not fully paid) causes the Random Forest to optimise overall accuracy at the expense of detecting defaulters.

**Business perspective:** For a loan risk model, **recall for class 1 is critical** (missing a defaulter costs money). In that context, the Decision Tree is arguably more useful despite its lower accuracy, because it correctly identifies more bad loans.

**Solution:** Apply class weighting (`class_weight='balanced'`) or resampling (SMOTE) to address the imbalance.

# Great Job!